In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Configurações padrão e opções de configuração da transpilação


<details>
<summary><b>Versões dos pacotes</b></summary>

O código nesta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar essas versões ou mais recentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Circuitos abstratos precisam ser transpilados porque os QPUs têm um conjunto limitado de portas base e não conseguem executar operações arbitrárias. A função do transpilador é transformar circuitos arbitrários para que possam ser executados em um QPU específico. Isso é feito traduzindo os circuitos para as portas base suportadas e introduzindo portas SWAP conforme necessário, para que a conectividade do circuito corresponda à do QPU.

Conforme explicado em [Transpilar com gerenciadores de passagem](transpile-with-pass-managers), você pode criar um [gerenciador de passagem](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) usando a função [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) e passar um circuito ou uma lista de circuitos para o método [run](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) para transpilá-los. Você pode chamar `generate_preset_pass_manager` passando apenas o nível de otimização e o backend, optando por usar os padrões para todas as outras opções, ou pode passar argumentos adicionais à função para ajustar a transpilação.

## Uso básico sem parâmetros
Neste exemplo, passamos um circuito e um QPU alvo para o transpilador sem especificar nenhum parâmetro adicional.

Crie um circuito e veja o resultado:

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# Create circuit to test transpiler on
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

# Add measurements to the circuit
qc.measure_all()

# View the circuit
qc.draw(output="mpl")

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg)

Transpile o circuito e veja o resultado:

In [2]:
from qiskit.transpiler import generate_preset_pass_manager

# Specify the QPU to target
backend = FakeSherbrooke()

# Transpile the circuit
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
transpiled_circ = pass_manager.run(qc)

# View the transpiled circuit
transpiled_circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg)

## Todos os parâmetros disponíveis
A seguir estão todos os parâmetros disponíveis para a função [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager). Há duas classes de argumentos: os que descrevem o alvo da compilação e os que influenciam o funcionamento do transpilador.

Todos os parâmetros, exceto `optimization_level`, são opcionais. Para mais detalhes, consulte a [documentação da API do Transpilador](https://docs.quantum.ibm.com/api/qiskit/transpiler#transpiler-api).

- `optimization_level` (int) - Quantidade de otimização a ser aplicada nos circuitos. Inteiro no intervalo (0 - 3). Níveis mais altos geram circuitos mais otimizados, ao custo de maior tempo de transpilação. Consulte [Definir o nível de otimização do transpilador](set-optimization) para mais detalhes.

### Parâmetros usados para descrever o alvo de compilação:
Esses argumentos descrevem o QPU alvo para execução do circuito, incluindo informações como o mapa de acoplamento do QPU (que descreve a conectividade dos qubits), as portas base suportadas pelo QPU e as taxas de erro das portas.

Muitos desses parâmetros são descritos em detalhes em [Parâmetros comumente usados para transpilação](common-parameters).

<details>
  <summary>
    **Parâmetros do QPU (`Backend`)**
  </summary>

**Parâmetros do backend** - Se você especificar `backend`, não precisa especificar `target` nem nenhuma outra opção de backend. Da mesma forma, se você especificar `target`, não precisa especificar `backend` nem nenhuma outra opção de backend.
- `backend` (Backend) - Se definido, o transpilador compila o circuito de entrada para este dispositivo. Se qualquer outra opção que impacte essas configurações for definida, como `coupling_map`, ela substitui as configurações do `backend`.
- `target` (Target) - Um alvo de transpilador de backend. Normalmente é especificado como parte do argumento backend, mas se você construiu manualmente um objeto Target, pode especificá-lo aqui. Isso substitui o alvo do `backend`.
- `backend_properties` (BackendProperties) - Propriedades retornadas por um QPU, incluindo informações sobre erros de porta, erros de leitura, tempos de coerência dos qubits, entre outros. Encontre um QPU que forneça essas informações executando `backend.properties()`.
- `timing_constraints` (Dict[str, int] | None) - Uma restrição opcional do hardware de controle sobre a resolução de tempo das instruções. Essa informação é fornecida pela configuração do QPU. Se o QPU não tiver nenhuma restrição na alocação de tempo das instruções, `timing_constraints` é `None` e nenhum ajuste é realizado. Um QPU pode reportar um conjunto de restrições, a saber:
    - `granularity`: Um valor inteiro que representa a resolução mínima de porta de pulso em unidades de dt. Uma porta de pulso definida pelo usuário deve ter uma duração que seja múltipla desse valor de granularidade.
    - `min_length`: Um valor inteiro que representa o comprimento mínimo de porta de pulso em unidades de dt. Uma porta de pulso definida pelo usuário deve ser mais longa que esse comprimento.
    - `pulse_alignment`: Um valor inteiro que representa a resolução temporal do tempo de início de instruções de porta. As instruções de porta devem iniciar em um tempo que seja múltiplo desse valor.
    - `acquire_alignment`: Um valor inteiro que representa a resolução temporal do tempo de início de instruções de medição. As instruções de medição devem iniciar em um tempo que seja múltiplo desse valor.
</details>

<details>
  <summary>
    **Parâmetros de layout e topologia**
  </summary>

- `basis_gates` (List[str] | None) - Lista de nomes de portas base para as quais desdobrar. Por exemplo ['u1', 'u2', 'u3', 'cx']. Se `None`, não desdobra.
- `coupling_map` (CouplingMap | List[List[int]]) - Mapa de acoplamento dirigido (possivelmente personalizado) como alvo no mapeamento. Se o mapa de acoplamento for simétrico, ambas as direções precisam ser especificadas. Os seguintes formatos são suportados:
    - Instância de CouplingMap
    - Lista - deve ser fornecida como uma matriz de adjacência, onde cada entrada especifica todas as interações dirigidas de dois qubits suportadas pelo QPU. Por exemplo: [[0, 1], [0, 3], [1, 2], [1, 5], [2, 5], [4, 1], [5, 3]]
- `inst_map` (List[InstructionScheduleMap] | None) - Mapeamento de operações de circuito para agendas de pulso. Se `None`, o `instruction_schedule_map` do QPU é usado.
</details>

### Parâmetros usados para influenciar o funcionamento do transpilador
Esses parâmetros afetam estágios específicos da transpilação. Alguns podem impactar múltiplos estágios, mas foram listados apenas em um estágio para simplicidade. Se você especificar um argumento, como `initial_layout` para os qubits que deseja usar, esse valor substitui todas as passagens que poderiam alterá-lo. Em outras palavras, o transpilador não mudará nada que você especificar manualmente. Para detalhes sobre estágios específicos, consulte [Estágios do transpilador](transpiler-stages).

<details>
  <summary>
    **Estágio de inicialização**
  </summary>

- `hls_config` (HLSConfig) - Uma classe de configuração opcional `HLSConfig` que é passada diretamente para a passagem de transformação `HighLevelSynthesis`. Essa classe de configuração permite especificar as listas de algoritmos de síntese e seus parâmetros para vários objetos de alto nível.
- `init_method` (str) - O nome do plugin a ser usado no estágio de inicialização. Por padrão, nenhum plugin externo é usado. Você pode ver uma lista de plugins instalados executando `list_stage_plugins()` com `init` como argumento do nome do estágio.
- `unitary_synthesis_method` (str) - O nome do método de síntese unitária a ser usado. Por padrão, `default` é usado. Você pode ver uma lista de plugins instalados executando `unitary_synthesis_plugin_names()`.
- `unitary_synthesis_plugin_config` (dict) - Um dicionário de configuração opcional que é passado diretamente para o plugin de síntese unitária. Por padrão, essa configuração não tem efeito porque o método padrão de síntese unitária não aceita configuração personalizada. Aplicar uma configuração personalizada só é necessário quando um plugin de síntese unitária é especificado com o argumento `unitary_synthesis`. Como isso é específico de cada plugin de síntese unitária, consulte a documentação do plugin para saber como usar essa opção.
</details>

<details>
  <summary>
    **Estágio de layout**
  </summary>

- `initial_layout` (Layout | Dict | List) - Posição inicial dos qubits virtuais nos qubits físicos. Se esse layout tornar o circuito compatível com as restrições do `coupling_map`, ele será usado. O layout final não tem garantia de ser o mesmo, pois o transpilador pode permutar qubits por meio de SWAPs ou outros meios. Para mais detalhes, consulte a [seção Layout inicial.](common-parameters#initial-layout)
- `layout_method` (str) - Nome da passagem de seleção de layout (`default`, `dense`, `sabre` e `trivial`). Também pode ser o nome do plugin externo a ser usado no estágio de layout. Você pode ver uma lista de plugins instalados executando `list_stage_plugins()` com `layout` para o argumento `stage_name`. O valor padrão é `sabre`.
</details>

<details>
  <summary>
    **Estágio de roteamento**
  </summary>

- `routing_method` (str) - Nome da passagem de roteamento (`basic`, `lookahead`, `default`, `sabre` ou `none`). Também pode ser o nome do plugin externo a ser usado no estágio de roteamento. Você pode ver uma lista de plugins instalados executando `list_stage_plugins()` com `routing` para o argumento `stage_name`. O valor padrão é `sabre`.
</details>

<details>
  <summary>
    **Estágio de tradução**
  </summary>

- `translation_method` (str) - Nome da passagem de tradução (`default`, `synthesis`, `translator`, `ibm_backend`, `ibm_dynamic_circuits`, `ibm_fractional`). Também pode ser o nome do plugin externo a ser usado no estágio de tradução. Você pode ver uma lista de plugins instalados executando `list_stage_plugins()` com `translation` para o argumento `stage_name`. O valor padrão é `translator`.
</details>

<details>
  <summary>
    **Estágio de otimização**
  </summary>

- `approximation_degree` (float, no intervalo 0-1 | None) - Parâmetro heurístico usado para aproximação de circuitos (1.0 = sem aproximação, 0.0 = aproximação máxima). O valor padrão é 1.0. Especificar `None` define o grau de aproximação como a taxa de erro reportada. Consulte a [seção Grau de aproximação](common-parameters#approx-degree) para mais detalhes.
- `optimization_method` (str) - O nome do plugin a ser usado no estágio de otimização. Por padrão, nenhum plugin externo é usado. Você pode ver uma lista de plugins instalados executando `list_stage_plugins()` com `optimization` para o argumento `stage_name`.
</details>

<details>
  <summary>
    **Estágio de agendamento**
  </summary>

- `scheduling_method` (str) - Nome da passagem de agendamento. Também pode ser o nome do plugin externo a ser usado no estágio de agendamento. Você pode ver uma lista de plugins instalados executando `list_stage_plugins()` com `scheduling` para o argumento `stage_name`.
  * 'as_soon_as_possible': Agenda instruções de forma gananciosa, o mais cedo possível em um recurso de qubit (apelido: `asap`).
  * 'as_late_as_possible': Agenda instruções tardiamente, ou seja, mantendo os qubits no estado fundamental quando possível (apelido: `alap`). Este é o padrão.
</details>

<details>
  <summary>
    **Execução do transpilador**
  </summary>

- `seed_transpiler` (int) - Define sementes aleatórias para as partes estocásticas do transpilador.
</details>

Os seguintes valores padrão são usados se você não especificar nenhum dos parâmetros acima. Consulte a [página de referência da API](../api/qiskit/transpiler_preset) do método para mais informações:

In [3]:
generate_preset_pass_manager(
    optimization_level=1,
    backend=None,
    target=None,
    basis_gates=None,
    coupling_map=None,
    initial_layout=None,
    layout_method=None,
    routing_method=None,
    translation_method=None,
    scheduling_method=None,
    approximation_degree=1.0,
    seed_transpiler=None,
    unitary_synthesis_method="default",
    unitary_synthesis_plugin_config=None,
    hls_config=None,
    init_method=None,
    optimization_method=None,
)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn how to [Set the optimization level](set-optimization).
    - Review more [Commonly used parameters](common-parameters).
    - Learn how to [Set the optimization level when using Qiskit Runtime](/docs/guides/runtime-options-overview).
    - Visit the [Transpile with pass managers](/docs/guides/transpile-with-pass-managers) topic.
    - For examples, see [Representing quantum computers](/docs/guides/represent-quantum-computers).
    - Learn [how to transpile circuits](/docs/guides/circuit-transpilation-settings) as part of the Qiskit patterns workflow using Qiskit Runtime.
    - Review the [Transpile API documentation](/docs/api/qiskit/transpiler).
</Admonition>